# Model 2 — Random Forest training, step by step
This adds a second ML model to the belt-based fall-detection project.
The original browser app and decision-tree model are preserved.

A Random Forest averages predictions from many trained decision trees. We use
100 trees and the same 12 sensor features as the first model.

**Subjects 5–6 were already inspected.** This is an exploratory comparison on
reused data, not a new independent final test. Select settings on subject 4,
then freeze them before this comparison run. Fresh participants are needed
for a new independent final evaluation.

Open this notebook inside FallDetection_Software and choose your existing Python
3.13 environment. Read saved outputs first. Running all cells again retrains
and replaces only the separate forest model and comparison reports.


## Step 1 — Import project functions
The existing core.py calculates sensor features. The new train_random_forest.py trains the second model.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'core.py').exists() and (ROOT / 'FallDetection_Software/core.py').exists():
    ROOT = ROOT / 'FallDetection_Software'
assert (ROOT / 'core.py').exists(), 'Open this notebook in the FallDetection_Software folder.'
sys.path.insert(0, str(ROOT))
from core import FEATURES, POLICY
from train_random_forest import PLAN, prepare_development, fit_candidates, freeze_forest, evaluate_models
from random_forest import load_forest, predict_forest
print('Project located. The original application uses the decision tree.')
print('Features:', len(FEATURES))


Project located. The original application uses the decision tree.
Features: 12


## Step 2 — Read the fixed experiment plan
Specify six forest configurations and three score thresholds before this run. Training uses subjects 1–3; validation uses 4; the reused comparison uses 5–6.

In [2]:
print(json.dumps(PLAN, indent=2))
print('Features:', ', '.join(FEATURES))

{
  "algorithm": "RandomForestClassifier",
  "n_estimators": 100,
  "max_depth_candidates": [
    5,
    8,
    null
  ],
  "min_samples_leaf_candidates": [
    2,
    5
  ],
  "threshold_candidates": [
    0.35,
    0.5,
    0.65
  ],
  "max_features": "sqrt",
  "class_weight": "balanced_subsample",
  "random_state": 42,
  "selection": "Validation F2, then precision, shallower depth, larger minimum leaf, higher threshold.",
  "split": {
    "train": [
      1,
      2,
      3
    ],
    "validation": [
      4
    ],
    "reused_test": [
      5,
      6
    ]
  },
  "comparison_status": "Exploratory comparison on previously inspected subjects 5-6. Not a new independent final test."
}
Features: acc_mean, acc_std, acc_min, acc_max, acc_range, gyro_mean, gyro_std, gyro_max, tilt_change, acc_last, gyro_last, acc_diff_max


## Step 3 — Load development windows
Each row of xtrain contains the 12 features for one eligible movement window. ytrain contains its fall/non-fall label. Labels and participant IDs do not enter the feature array.

In [3]:
training, validation = prepare_development(ROOT)
xtrain, ytrain = training[2:]
xval, yval = validation[2:]
print('xtrain shape:', xtrain.shape)
print('Training fall windows:', int(ytrain.sum()))
print('Validation fall windows:', int(yval.sum()))


Training: 3237 windows; validation: 1147 windows.
xtrain shape: (3237, 12)
Training fall windows: 131
Validation fall windows: 30


## Step 4 — Train candidates and select using validation
Inside fit_candidates(), the actual training operation is:
```python
clf = RandomForestClassifier(n_estimators=100, ...)
clf.fit(xtrain, ytrain)
```
Candidate forests are fitted on training people only. Scores and thresholds
are compared using validation F2, then precision and the specified tie rules.
This selection code does not use comparison-set measurements.


In [4]:
clf, threshold, candidates = fit_candidates(training, validation)
print('Selected model:', type(clf).__name__)
print('Trees:', clf.n_estimators)
print('Depth:', clf.max_depth)
print('Minimum leaf size:', clf.min_samples_leaf)
print('Score threshold:', threshold)


Fitted 100 trees: depth=5, minimum leaf=2
Fitted 100 trees: depth=5, minimum leaf=5
Fitted 100 trees: depth=8, minimum leaf=2
Fitted 100 trees: depth=8, minimum leaf=5
Fitted 100 trees: depth=None, minimum leaf=2
Fitted 100 trees: depth=None, minimum leaf=5
Selected model: RandomForestClassifier
Trees: 100
Depth: 5
Minimum leaf size: 5
Score threshold: 0.65


## Step 5 — Freeze the selected forest
The portable JSON contains the fitted trees. It is separate from trained_tree.json. Forest scores are averages of learned leaf scores, not calibrated real-world probabilities.

In [5]:
forest = freeze_forest(clf, threshold, candidates, ROOT)
print('Saved model/trained_forest.json')
print('Model ID:', forest['model_id'])

Selected settings frozen: depth=5, leaf=5, threshold=0.65
Saved model/trained_forest.json
Model ID: fall-forest-v1


## Step 6 — Compare actual predictions
Now load reused comparison subjects 5–6. Verify that exported forest scores match scikit-learn within 1e-12 on every evaluated window. Load the existing decision tree without retraining it.

In [6]:
report = evaluate_models(clf, forest, training, validation, ROOT)
comparison = pd.read_csv(ROOT / 'reports/random_forest_comparison.csv')
columns = ['method', 'accuracy', 'precision', 'recall', 'f1', 'false_event_groups_in_nonfall_recordings']
print(comparison.loc[comparison['split'].eq('reused_test'), columns].round(4).to_string(index=False))


Evaluated train: 3237 windows; portable scores match sklearn.
Evaluated validation: 1147 windows; portable scores match sklearn.
Evaluated reused_test: 2111 windows; portable scores match sklearn.
       method  accuracy  precision  recall    f1  false_event_groups_in_nonfall_recordings
decision_tree    0.9588     0.3740  0.8214 0.514                                        15
random_forest    0.9730     0.4921  0.5536 0.521                                         4


## Step 7 — Understand the tradeoff
Accuracy is dominated by non-fall windows in this subset. Compare recall, precision, missed fall windows and false alarms. Window counts and grouped event counts use different units.

In [7]:
for method, result in report['splits']['reused_test'].items():
    measured = result['window_metrics']
    (tn, fp), (fn, tp) = measured['confusion_matrix']
    print(f'{method}: {tp} fall windows detected; {fn} missed; {fp} false-positive windows.')
    print(f"  Balanced accuracy {measured['balanced_accuracy']:.2%}; F1 {measured['f1']:.3f}")


decision_tree: 46 fall windows detected; 10 missed; 77 false-positive windows.
  Balanced accuracy 89.20%; F1 0.514
random_forest: 31 fall windows detected; 25 missed; 32 false-positive windows.
  Balanced accuracy 76.90%; F1 0.521


## Step 8 — Try sensor inputs with the second model
These are known training excerpts for demonstrating inference, not new validation. Edit a copy of a CSV or change csv_path to your own sequence. The schema and timing rules match the first app.

In [8]:
loaded_forest = load_forest(ROOT / 'model/trained_forest.json')
for kind in ['walking', 'fall']:
    csv_path = ROOT / f'examples/{kind}_window.csv'
    predictions = predict_forest(loaded_forest, pd.read_csv(csv_path))
    print(kind, 'training excerpt:')
    for prediction in predictions:
        label = 'Suspected fall' if prediction['suspected_fall'] else 'No fall predicted'
        print(f"  {prediction['time_s']:.2f} s: {label}; score {prediction['fall_score']:.3f}")


walking training excerpt:
  1.02 s: No fall predicted; score 0.109
  1.27 s: No fall predicted; score 0.097
fall training excerpt:
  1.00 s: Suspected fall; score 0.971
  1.28 s: Suspected fall; score 0.972


## What to show in the presentation
- A single decision tree versus an ensemble of 100 trees.
- Identical sensor inputs, features and participant protocol.
- Actual precision/recall tradeoffs rather than claiming every metric improved.
- A new training script, fitted forest and comparison report.
- Reused comparison data and the need for fresh final evaluation participants.

The web app continues using the original decision tree. This notebook and
random_forest.py demonstrate the second model separately. Hardware, mobile
integration and on-device forest timing are not implemented by this add-on.

Sources: [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html),
[UP-Fall dataset](https://sites.google.com/up.edu.mx/har-up/),
[dataset paper](https://doi.org/10.3390/s19091988).
